In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform = transform)
test_set = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform = transform)

train_loader = DataLoader(train_set, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_set, batch_size = 10000, shuffle = False)

In [3]:
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(

            torch.nn.Linear(784, 256),
            torch.nn.GELU(),
            
            torch.nn.Linear(256, 128),
            torch.nn.GELU(),

            torch.nn.Linear(128, 64),
            torch.nn.GELU(),

            torch.nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

In [4]:
model = MNISTClassifier()

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.05)

In [8]:
def training(model, train_loader, loss_function, optimizer):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = loss_function(output, target)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
        
        if batch_idx % 100 == 0 and batch_idx > 0:
            avg_loss = running_loss / 100
            accuracy = 100.0 * (correct / total)
            print(f' [{batch_idx * 64} / {60000}] '
                  f'Loss: {avg_loss:.3f} | Train Accuracy: {accuracy:.2f}%')
            running_loss = 0.0 

In [6]:
def evaluate(model, test_loader):
    model.eval()

    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
           
    return 100.0 * (correct / total)

In [7]:
num_epochs = 5
for epoch in range(num_epochs):
    print(f'\nEpoch: {epoch + 1}')
    training(model, train_loader, loss_function, optimizer)
    accuracy = evaluate(model, test_loader)
    print(f'Dev Accuracy:{accuracy:.2f}')


Epoch: 1
 [6400 / 60000] Loss: 2.179 | Accuracy: 30.14%
 [12800 / 60000] Loss: 0.833 | Accuracy: 52.85%
 [19200 / 60000] Loss: 0.455 | Accuracy: 63.93%
 [25600 / 60000] Loss: 0.369 | Accuracy: 70.20%
 [32000 / 60000] Loss: 0.288 | Accuracy: 74.48%
 [38400 / 60000] Loss: 0.280 | Accuracy: 77.36%
 [44800 / 60000] Loss: 0.261 | Accuracy: 79.52%
 [51200 / 60000] Loss: 0.220 | Accuracy: 81.23%
 [57600 / 60000] Loss: 0.209 | Accuracy: 82.62%
Test Accuracy:94.15

Epoch: 2
 [6400 / 60000] Loss: 0.174 | Accuracy: 94.79%
 [12800 / 60000] Loss: 0.157 | Accuracy: 94.99%
 [19200 / 60000] Loss: 0.168 | Accuracy: 94.91%
 [25600 / 60000] Loss: 0.147 | Accuracy: 95.10%
 [32000 / 60000] Loss: 0.138 | Accuracy: 95.26%
 [38400 / 60000] Loss: 0.132 | Accuracy: 95.36%
 [44800 / 60000] Loss: 0.146 | Accuracy: 95.38%
 [51200 / 60000] Loss: 0.139 | Accuracy: 95.43%
 [57600 / 60000] Loss: 0.129 | Accuracy: 95.49%
Test Accuracy:95.89

Epoch: 3
 [6400 / 60000] Loss: 0.100 | Accuracy: 97.17%
 [12800 / 60000] Loss